# Euclid-like dataset generation (complex simulator)

This notebook demonstrates how to generate a small euclid-like dataset using the
`ComplexSimulator` with NISP-like noise and multi-order dispersion. It also
provides inspection and visualization utilities so you can quickly validate
that the simulated HDF5 files contain the expected groups and data shapes.

Notebook outline
----------------
1. Generate and save a small dataset (or skip generation if file already exists).
2. Load and inspect metadata and dataset structure.
3. Visualize example spectrograms, the direct (broadband) image, and
   a few spectral slices from the ground-truth cube.

Notes
-----
- The miniature configuration uses `n_spectral_pixels=128` so this notebook is
  fast to run on a developer machine.  For full Euclid realism use the complex
  simulator with higher spectral sampling and realistic SEDs (synphot).
- All generated data are stored in compressed HDF5 files under `data/raw/`.

In [ ]:
from pathlib import Path
import numpy as np
from spectangle.simulations.complex import ComplexSimulator
from spectangle.simulations.io import save_simulation

# Output path
output = Path('data/raw/complex_euclid_50s.h5')
output.parent.mkdir(parents=True, exist_ok=True)

# Create simulator configured for a miniature, euclid-like run
sim = ComplexSimulator(
    n_sources=20,
    image_shape=(128, 128),
    n_spectral_pixels=128,
    add_noise=True,
    use_realistic_seds=False,  # set True if synphot + realistic SEDs are available
    seed=123,
)

# Generate only if the file doesn't already exist to avoid expensive re-generation
if not output.exists():
    samples = []
    for i in range(50):
        samples.append(sim.generate_one(np.random.default_rng(i + 1000)))
    save_simulation(samples, output, metadata={'note': 'euclid-like miniature dataset (no full realism)'})
    print('Wrote:', output)
else:
    print('Dataset already exists:', output)

## Inspecting the saved HDF5 file
Load the saved file, print metadata and show the available groups. This helps to
confirm padding, spectral dimension and that spectrograms/direct image shapes
are consistent with the forward model.

In [ ]:
# Cell: load and inspect
from spectangle.simulations.io import load_simulation

data = load_simulation(output)
print('Top-level keys:', list(data.keys()))
print('\nMetadata:')
for k, v in data['metadata'].items():
    print(f'  {k}:', v)

n_samples = len(data['samples'])
print('\nNumber of samples loaded:', n_samples)

# Show shapes for the first sample
s0 = data['samples'][0]
print('\nSample 0 keys:', list(s0.keys()))
print('cube shape:', s0['cube'].shape)
print('wavelengths shape:', data['wavelengths'].shape)
print('spectrograms shape:', s0['spectrograms'].shape)
if 'direct_image' in s0 and s0['direct_image'] is not None:
    print('direct_image shape:', s0['direct_image'].shape)

# Expose some handy variables
wavelengths = data['wavelengths']
cube0 = s0['cube']  # (n_lambda, ny, nx)
specs0 = s0['spectrograms']  # (4, H_spec, W_spec)
direct0 = s0.get('direct_image')

## Visualize a sample
We will plot:
- The 4 K-sequence spectrograms (padded detector images).
- The direct (broadband) image if present.
- A 2x2 grid showing three spectral slices from the cube (blue/center/red)
  and the wavelength-integrated (broadband) cube for comparison.

These plots help verify alignment between the direct image and the dispersed
spectrograms and to get intuition about how spectra map across the detector.

In [ ]:
import matplotlib.pyplot as plt

# Helper to plot the 4 spectrograms
fig, axs = plt.subplots(1, 5, figsize=(18, 4))
for k in range(4):
    ax = axs[k]
    im = ax.imshow(specs0[k], origin='lower', cmap='inferno')
    ax.set_title(f'Spectrogram {k+1}')
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

# Direct image in the last panel if present
ax = axs[4]
if direct0 is not None:
    im = ax.imshow(direct0, origin='lower', cmap='gray')
    ax.set_title('Direct (broadband)')
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
else:
    ax.text(0.5, 0.5, 'No direct image', ha='center', va='center')
    ax.axis('off')

plt.suptitle('Sample 0: Spectrograms + Direct image')
plt.tight_layout()
plt.show()

# Plot spectral slices from the cube
n_lambda = cube0.shape[0]
idx_blue = int(n_lambda * 0.1)
idx_center = int(n_lambda * 0.5)
idx_red = int(n_lambda * 0.9)

fig, axs = plt.subplots(1, 4, figsize=(16, 4))
# Blue slice
axs[0].imshow(cube0[idx_blue], origin='lower', cmap='viridis')
axs[0].set_title(f'Blue slice (λ={int(wavelengths[idx_blue])} Å)')
axs[0].axis('off')
# Center slice
axs[1].imshow(cube0[idx_center], origin='lower', cmap='viridis')
axs[1].set_title(f'Center slice (λ={int(wavelengths[idx_center])} Å)')
axs[1].axis('off')
# Red slice
axs[2].imshow(cube0[idx_red], origin='lower', cmap='viridis')
axs[2].set_title(f'Red slice (λ={int(wavelengths[idx_red])} Å)')
axs[2].axis('off')
# Broadband (wavelength integrated)
broad = cube0.sum(axis=0)
im = axs[3].imshow(broad, origin='lower', cmap='gray')
axs[3].set_title('Broadband (cube sum)')
axs[3].axis('off')
fig.colorbar(im, ax=axs[3], fraction=0.046, pad=0.02)
plt.suptitle('Spectral slices from ground-truth cube (sample 0)')
plt.tight_layout()
plt.show()

## Inspect individual spectra
Find the brightest pixel in the broadband cube (likely a simulated point source)
and plot its spectrum alongside the average field spectrum. This helps check
that the SED shapes and noise behaviour are reasonable.

In [ ]:
# Find brightest pixel
broad = cube0.sum(axis=0)
iy, ix = np.unravel_index(np.argmax(broad), broad.shape)
print('Brightest pixel at (y,x)=', (iy, ix))

spec_at_max = cube0[:, iy, ix]
avg_spec = cube0.mean(axis=(1, 2))

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(wavelengths, spec_at_max, label='spectrum at brightest pixel')
ax.plot(wavelengths, avg_spec, label='average spectrum', alpha=0.7)
ax.set_xlabel('Wavelength [Å]')
ax.set_ylabel('Flux (arbitrary units)')
ax.set_title('Example spectra (sample 0)')
ax.legend()
plt.tight_layout()
plt.show()

## Quick checks and next steps
- Confirm that spectrogram padding (pad_x, pad_y) matches expectations
  (see metadata printed above).
- If the direct image is misaligned relative to the broadband cube, review the
  ForwardModel padding and PSF configuration.

Next steps:
- Use the generated HDF5 with the training notebooks (miniature or euclid_like)
  to train UNet / ViT / PINN models. The PINN notebook demonstrates how to
  use the differentiable forward model and physics loss term.
- For visualization during training add callbacks that save example reconstructions
  (reprojected via the forward model) to monitor physics residuals.